---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-38: Hands-on Examples of <b>Vectorized and Vectorless RAG</b></h1>

# Learning agenda of this notebook  

1. Overview of Retrieval-Augmented Generation (RAG)
2. RAG Architecture
3. A Step-by-Step Example of a Basic RAG Application
4. Chatbot that extracts a youtube video transcript and you can ask Qs from that video using RAG
5. RAG Types & Techniques
6. Vector RAG vs Vectorless RAG

### [Retrieval-Augmented Generation for Large Language Models: A Survey (Mar 2024)](https://arxiv.org/pdf/2312.10997)
### [Enhancing RAG: A Study of BestPractices](https://github.com/arifpucit/Generative-and-Agentic-AI/blob/main/Research%20Articles/39-Enhancing_RAG_A_Study_of_Best_Practices.pdf)
### [Multi-RAG: A Multimodal Retrieval-Augmented Generation System for Adaptive Video Understanding (Jun 2025)](https://arxiv.org/pdf/2505.23990v2)

# <span style='background :lightgreen' >Recap: Prompt Engineering vs RAG vs Fine-Tuning</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">There are three ways using which you can get better output/response from a Large Language Model</h3>


<div style="text-align: center;">
    <img src="../images/RAGc.png" style="max-width: 90%; margin-bottom: 15px;">
</div>


### (i) Prompt Engineering: 
- Ask the model a query that better specifies  what we are looking for.
- For example if you ask ChatGPT a question "Who is Muhammad Arif Butt?", he might be knowing many persons in the world with this name.
- So if you ask a specific question like "Who is Muhammad Arif Butt at Punjab University?", the model might be giving you a better response.
- **Prompt Engineering is the art of crafting well-structured input prompts to guide LLMs towards desired outputs.**
- It's the fastest and most cost-effective way to improve model performance through strategic questioning, examples, and instruction formatting.

### (ii) RAG (Retrieval-Augmented Generation):
- **RAG `retrieves` external knowldege and `augment` the original prompt with the retrieved information to `generate` better results.**
-  Formally speaking, RAG is a technique that combines LLMs with external knowledge retrieval systems to provide accurate, up-to-date information beyond the model's training data.
-  It allows models to access real-time information and domain-specific knowledge without re-training or fine-tuning.
-  The possible external knowledge retrieval systems that RAG models can use:
    - Vector Databases (Pinecone, Chroma, FAISS, MongoDB, Elasticsearch)
    - APIs and Web Services (Web Scraping, Google/Bing Search API for real-time web information, Wikipedia API, Custom APIs to search Internal company databases)
    - Specialized Knowledge Sources (Code Repositories (GitHub, GitLab), Structured Data (SQL databases, data warehouses, CSV files), Enterprise Systems (SharePoint, Confluence, Notion) and  academic paper repositories)

### (iii) Fine-tuning:
- **The process of training a pre-trained LLM on specific datasets to specialize it for particular tasks, domains, or writing styles.**
- In fine-tuning we are actually making adjustments to the model's weights using the specialized dataset(s).
- This process typically a supervised learning, where we provide input-output pairs that demonstrate the kind of response that we want.
- For example, if we are fine-tuning a base model for customer support purpose, we might provide thousands of examples of customer queries along with their correct technical responses. The model adjusts its weights through back-propagation to minimize the difference between its predicted outputs and the targeted responses.
- In comparison to RAG-based models, it is much faster at response time, because it does not need to search for external data on each query. Moreover, you donot need to maintain a separate vector database. However, like no free lunch fine-tuning has its own limitations.
    - Fine-tuning requires expensive GPU resources and compute time (hours/days)
    - Fine-tuning needs technical ML expertise
    - Fine-tuning needs large, high-quality labeled datasets (thousands of examples)
    - Fine-tuning requires careful data curation and preprocessing
    - Fine-tuning creates a static model - changes require retraining from scratch
    - Fine-tuning locks you into specific behaviors

# <span style='background :lightgreen' >1. Overview of Retrieval-Augmented Generation (RAG)</span>

## a. What is RAG?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">RAG is a powerful technique that enhances AI models by combining their generation capabilities with external knowledge retrieval.</h3>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">RAG allows models to access and use external knowledge sources to generate more accurate and informed responses. RAG systems first retrieve relevant documents from a knowledge base, then use this information as context for generating answers.</h3>

- **Real-World Analogy:**
    - Traditional Language Models are like students taking closed-book exam - they can only use what they memorized.
    - RAG-enabled Models are like students taking open-book exam - they can reference materials to provide more accurate, detailed and up-to-date answers.

- **RAG:**
    - **Retrieval:** Finding relevant information using semantic search from a vector database.
    - **Augmented:** Enhancing user query with the retrieved context.
    - **Generation:** Generating response to user query.

## b. How Naive RAG work:
- **RAG (Retrieval-Augmented Generation)** is a technique where an LLM is grounded in an external knowledge source at query time, rather than relying solely on its training data. 

In [1]:
from IPython.display import HTML
HTML("""
<div style="text-align: center;">
  <video id="myvideo" width="700" height="400" controls style="display: inline-block;">
    <source src="../images/How-data-is-stored.mp4" type="video/mp4">
  </video>
</div>

<script>
  var video = document.getElementById('myvideo');
  video.playbackRate = 0.5;  // Change playbackrate to any value like 0.5, 1.0, 1.5, 2.0, etc.
</script>
""")

# <span style='background :lightgreen' >2. RAG Architecture</span>

<div style="text-align: center;">
    <img src="../images/RAGa.png" style="max-width: 70%; margin-bottom: 15px;">
</div>


<img align="right" width="1000" src="../images/r2.png"  >

### (i) Document Ingestion Phase: Building Knowledge Base
1. An AI Engineer prepares the client data (for example, procedure manuals, product documentation, or help desk tickets, etc.) during Data Preprocessing and make it suitable for model augmentation. Transformations might include simple format conversions such as converting PDF documents to text, or more complex transformations such as translating complex table structures into if-then type statements using **Document Laoders**. Enrichment may include expanding common abbreviations, adding meta-data such as currency information, and other additions to improve the relevancy of search results. He may also break down long documents into smaller chunks that are easier to process and search through using **Text Splitters**.
2. The AI Engineer then use an appropriate **embedding model** convert the source data into a series of vectors.
3. The generated embeddings are then stored in a vector database such as FAISS, Chroma, Pinecone, or Milvus for fast similarity-based search.
### (ii) Query Processing Phase
4. End-users now interact with a GenAI enabled application and enter a query, which gets converted into the same vector format as stored documents using same Embedding model. This is done by the **Retriever** component.
5. The GenAI application performs a search on the vector database to obtain the top most (we call this top K) vectors/chunks that most closely match the user's query vector. These top K vectors are unembedded. (cosine similarity, Euclidean distance, Dot product etc)
6. The top K chunks, along with the user query are sent to the LLM. This is called context augmentation.
### (iii)Generation Phase: Answering Questions
7. The LLM returns a human-like response based on the user's query, prompt, and context information which is presented to the end-user.

# <span style='background :lightgreen' >3. A Step-by-Step Example  of a Basic RAG Application</span>

## Step 1 (Document Loading): Load documents from various sources

In [2]:
from langchain_community.document_loaders import DirectoryLoader    # Loads multiple documents from a director
from langchain_community.document_loaders import  TextLoader        # Loads text files (.txt) into Document objects

# load all the text files from the directory
loader = DirectoryLoader(
                        "../data/",               # Path to the directory containing the text files
                        glob="**/*.txt",                     # Pattern to match files recursively (all .txt files in all subfolders)
                        loader_cls= TextLoader,              # Use TextLoader to handle each matched file
                        loader_kwargs={'encoding': 'utf-8'} # Additional arguments passed to TextLoader (UTF-8 encoding)
                    )
# The loader() METHOD reads every text file matching the pattern and returns a list of LangChain Document objects, each containing: page_content and metadata
documents = loader.load()  

print(f"Loaded {len(documents)} documents")         # Print total number of documents loaded
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Metadata: {doc.metadata}")                    # Print metadata (e.g., file path)
    print(f"  Length: {len(doc.page_content)} characters")  # Print number of characters in document
    print(f"  File Content: {doc.page_content[:100]}...")   # Print first 100 characters as a preview

Loaded 6 documents

Document 1:
  Metadata: {'source': '../data/cricket.txt'}
  Length: 4591 characters
  File Content: Cricket: The Gentleman's Game

Cricket is a bat-and-ball sport played between two teams of eleven pl...

Document 2:
  Metadata: {'source': '../data/names.txt'}
  Length: 1176 characters
  File Content: Cricket in Pakistan has always been more than just a sport—it’s a source of national pride and unity...

Document 3:
  Metadata: {'source': '../data/python_intro.txt'}
  Length: 489 characters
  File Content: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...

Document 4:
  Metadata: {'source': '../data/machine_learning.txt'}
  Length: 575 characters
  File Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...

Document 5:
  Metadata: {'source': '../data/proposal.txt'}
  Length: 484 characters
  File Content: Project Proposal: RAG Implementation

Executive Su

## Step 2 (Document Splitting): Break documents into smaller chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Import the RecursiveCharacterTextSplitter class

# Initialize RecursiveCharacterTextSplitter class
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Each chunk will contain up to 500 characters
    chunk_overlap=50,     # Each chunk will overlap the next one by 50 characters that helps retain context across chunks
    length_function=len,  # Function used to measure text length (default: Python's len)
    separators=[" "]      # Defines where to split text — tries to split at spaces, but if not possible, it goes deeper recursively
)


# Each document in 'documents' is split into smaller, overlapping segments to make them suitable for embedding or LLM input.
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print("-" * 60)  # separator line
for i, chunk in enumerate(chunks, start=1):
    print(f"--- First 100 characters of Chunk {i} ---")        # Show which chunk this is
    print(chunk.page_content[:100])                            # Print first 100 characters of the chunk
    print(f"(Length: {len(chunk.page_content)} characters)")   # Show actual length of the chunk in characters
    print("-" * 60)                                            # # Print separator line between chunks

Created 22 chunks from 6 documents
------------------------------------------------------------
--- First 100 characters of Chunk 1 ---
Cricket: The Gentleman's Game

Cricket is a bat-and-ball sport played between two teams of eleven pl
(Length: 495 characters)
------------------------------------------------------------
--- First 100 characters of Chunk 2 ---
sport of England. The expansion of the British Empire led to cricket being played overseas and by th
(Length: 493 characters)
------------------------------------------------------------
--- First 100 characters of Chunk 3 ---
the bowling team tries to dismiss the batsmen and restrict runs.

Equipment and Field
Players use a 
(Length: 487 characters)
------------------------------------------------------------
--- First 100 characters of Chunk 4 ---
South Africa, West Indies, and Zimbabwe. The International Cricket Council (ICC) governs the sport g
(Length: 496 characters)
---------------------------------------------------------

## Step 3: Create In Memory Vector Store using Chroma, add the chunks to it, and display the chunks along with their embeddings

In [4]:
from langchain_chroma import Chroma # Import Chroma from langchain_chroma (newer, recommended import)
from langchain_huggingface import HuggingFaceEmbeddings # HuggingFaceEmbeddings class provides an interface to use pre-trained Hugging Face sentence-transformer models for generating text embeddings


# Initialize the Local Embedding Model using the `HuggingFaceEmbeddings` class that wraps Sentence Transformers (from Hugging Face)
# The returned object provides a simple interface to generate embeddings of text that can later be stored in a vector db or used for similarity search.
local_embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")      #"all-mpnet-base-v2"
local_embedder


# Chroma class is a vector store wrapper in LangChain for ChromaDB. It provides a high-level Python interface for storing document embeddings, performing similarity searches and integrating with LLM pipelines
vectorstore = Chroma(
    collection_name = "rag_collection",      # Name of the collection inside Chroma
    embedding_function = local_embedder,     # The embedding model used to generate vector representations
    #persist_directory = ,   # Directory where database files are saved
    collection_metadata={"hnsw:space": "cosine"}  # For Euclidean (L2) distance: "l2", For Inner product: "ip"
)

#vectorstore.delete_collection()  # If the collection already exist then use this
vectorstore.reset_collection() # This will allow you to repeatedly execute this cell code and it will not add the documents again and again
# Add the all the documents  to the In Memory Chroma db using its `add_documents()` method that computes embeddings for each document and stores them
vectorstore.add_documents(documents = chunks)

#data = vectorstore.get()                      # returns a dictionary containing 'ids', 'embeddings', 'documents', and 'metadatas'
data = vectorstore.get(include=["documents", "metadatas", "embeddings"]) # returns a dictionary containing 'documents', 'metadatas', and  `embeddings`

print("\033[1m=== Displaying stored documents ===\033[0m")
for i, (doc, meta, emb) in enumerate(zip(data["documents"], data["metadatas"], data["embeddings"]),start=1):
    print(f"Document {i}")
    print(f"Metadata: {meta}")
    print(f"Content: {doc}")
    print(f"Embedding (first 5 values): {emb[:5]}...")
    print("-" * 60)

=== Displaying stored documents ===
Document 1
Metadata: {'source': '../data/cricket.txt'}
Content: Cricket: The Gentleman's Game

Cricket is a bat-and-ball sport played between two teams of eleven players on a field at the center of which is a 22-yard pitch with a wicket at each end. The game is played by 120 million players in many countries, making it the world's second most popular sport.

History of Cricket
Cricket was first played in southern England in the 16th century. By the end of the 18th century, it had developed into the national sport of England. The expansion of the British
Embedding (first 5 values): [ 0.08470259  0.03039418 -0.04517036 -0.11391057 -0.03823341]...
------------------------------------------------------------
Document 2
Metadata: {'source': '../data/cricket.txt'}
Content: sport of England. The expansion of the British Empire led to cricket being played overseas and by the mid-19th century the first international matches were being held.

Basic Rules and F

## Step 4 (Query Processing): Create a Retriever and pass the User Query (string) to its `invoke()` method
- Retrieve top-k most relevant Chunks from the vector store 

In [5]:
query = "Who is Dr. Muhammad Arif Butt?"

# Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",           # Type of search to perform: 'similarity', 'mmr', etc.
    search_kwargs={"k": 4},             # Dictionary of search parameters, default is none
    tags=None,             # List of tags for tracing/monitoring
    metadata=None,         # Dictionary of metadata
    verbose=False          # Enable verbose logging
)

results = retriever.invoke(query)     # The retriever.invokesimilarity_search() returns list containing Document objects only
unembedded_texts = []   # Initialize an empty list to store plain text from the retrieved Document objects
print(f"\n🔍 Query: '{query}'")
# Display results
for i, doc in enumerate(results, 1):
    print(f"   Content {i}: {doc.page_content[:150]}...")
    print(f"   Metadata{i}: {doc.metadata}\n")
    unembedded_texts.append(doc.page_content)        # Add it to our list for potential later use
print(unembedded_texts)


🔍 Query: 'Who is Dr. Muhammad Arif Butt?'
   Content 1: conditions.

With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, ...
   Metadata1: {'source': '../data/arif_bio.txt'}

   Content 2: Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. His expertise in...
   Metadata2: {'source': '../data/arif_bio.txt'}

   Content 3: Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He h...
   Metadata3: {'source': '../data/arif_bio.txt'}

   Content 4: Co-Founder of Tbox Solutionz. In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit develop...
   Metadata4: {'source': '../data/arif_bio.txt'}

['conditions.\n\nWith over 33 years of experience in teaching and management, Dr. But

## Step 5 (Context Augmentation): Build a prompt using the original query + top-K texts

In [6]:
# Step 1: Build the context section for the LLM prompt
# Combine the top retrieved text chunks (unembedded_texts) into a single context block that will guide the LLM’s response. Each chunk is labeled so the model can reference them (1/2/3).
context_parts = []
for i, t in enumerate(unembedded_texts, start=1):
    context_parts.append(f"--- Retrieved Document {i} ---\n{t}\n") # Add each retrieved document with a label for easy referencing

# Join all retrieved document texts into one large string, separated by line breaks for clarity.
context = "\n".join(context_parts).strip()


# Step 2: Construct the full prompt for the language model containing system role, context, user query and specific instructions as to how the model should respond
prompt = (
    "You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.\n\n"
    f"CONTEXT:\n{context}\n\n"        # Inject the retrieved document text here
    f"USER QUERY:\n{query}\n\n"    # The user's original question
    "INSTRUCTIONS: Answer concisely and cite which retrieved document (1/2/3) you used for any factual statement if relevant.\n\nAnswer:"
)


# Step 3: Display the constructed prompt for review
print(prompt)

You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.

CONTEXT:
--- Retrieved Document 1 ---
conditions.

With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.

Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. In recent years, he

--- Retrieved Document 2 ---
Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. His expertise in cybersecurity and emerging technologies enables him to contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.

--- Retr

## Step 6 (Response Generation): Send the original query + top-K texts to an LLM of your Choice to generate answer using context

In [7]:
from dotenv import load_dotenv              # load_dotenv() method is used to securely load API keys and environment variables from a .env file
from langchain_openai import ChatOpenAI       # ChatOpenAI is LangChain wrapper for OpenAI-compatible chat models (also supports Groq and others)
import os

# Load environment variables from the .env file
load_dotenv('../keys/.env', override=True) 
groq_api_key = os.getenv('GROQ_API_KEY')

# Initialize the LLM (Groq-hosted model via OpenAI-compatible API)
model = ChatOpenAI(
    model="llama-3.3-70b-versatile",  
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",  # Point to Groq API
    temperature=0.7,                            # temperature: controls creativity (0.0 = deterministic, 1.0 = more creative)    
    max_tokens=512                              # max_tokens: limits the length of the generated response.
)

# Send the constructed prompt to the model using the .invoke() method that sends the text prompt to the model and returns a structured response object.
response = model.invoke(prompt) 

# Display the model's answer
print(response.content)    # The `content` attribute contains the actual text generated by the model.

Dr. Muhammad Arif Butt is an accomplished Assistant Professor (Retrieved Document 3) with over 33 years of experience in teaching and management (Retrieved Document 1 and 3), and a technology entrepreneur (Retrieved Document 1). He has expertise in areas such as cybersecurity, artificial intelligence, and operating systems (Retrieved Document 1 and 2).


# <span style='background :lightgreen' >4. Chatbot that extracts a youtube video transcript and you can ask Qs from that video using RAG</span>
#### https://www.youtube.com/watch?v=ndl79-VDl50

```bash
uv add  youtube-transcript-api
```

## Step 1 (Document Loading): Load documents from YouTube Video

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

video_id = "ndl79-VDl50" # Arif youtube video (Only the ID, not full URL)

api = YouTubeTranscriptApi()
    
# Get transcript list
transcript_list = api.list(video_id)

# Pick English transcript
transcript = transcript_list.find_transcript(['en'])

# Fetch transcript segments
transcript_data = transcript.fetch()

# NOTE: transcript_data in v1.2.3 returns a list of FetchedTranscriptSnippet objects
# Each snippet has attributes like 'text', 'start', 'duration'
transcript_text = " ".join([t.text for t in transcript_data])

print("✅ Transcript fetched successfully!\n")
#print(transcript_text[:500])  # Print preview
print(transcript_text)  # Print preview

✅ Transcript fetched successfully!

bismillah ar-rahman rahim assalamu alaikum dear students I welcome you to this series of lectures for linux shell commands these are long awaited by my students operating system as well as system programming the main objective of the series of lectures is to acquaint the students of undergraduate operating system course to map the concepts that to study in the class using Linux tools it is also useful for the students who are learning Linux as system administrator students in this course in this series of lectures basically we will start with the concepts of UNIX shell then we will see how processes are managed by the UNIX shell how processes are created how the priorities are managed how they are scheduled how they are terminated we will see in detail how information about all the running processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux particularly th

## Step 2 (Document Splitting): Break documents into smaller chunks

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("\nRECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Try these separators in order
    chunk_size=1000,          # Target maximum: 100 characters per chunk
    chunk_overlap=200,        # Share 20 characters between consecutive chunks
    length_function=len
)

chunks = recursive_splitter.split_text(transcript_text)
# Print each chunk one by one
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}:")
    print(f"'{chunk}'")
    print(f"Length: {len(chunk)} characters")
    print("-" * 50)


RECURSIVE CHARACTER TEXT SPLITTER
Chunk 1:
'bismillah ar-rahman rahim assalamu alaikum dear students I welcome you to this series of lectures for linux shell commands these are long awaited by my students operating system as well as system programming the main objective of the series of lectures is to acquaint the students of undergraduate operating system course to map the concepts that to study in the class using Linux tools it is also useful for the students who are learning Linux as system administrator students in this course in this series of lectures basically we will start with the concepts of UNIX shell then we will see how processes are managed by the UNIX shell how processes are created how the priorities are managed how they are scheduled how they are terminated we will see in detail how information about all the running processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux partic

## Step 3: Create In Memory Database using Chroma, add the chunks to it, and display the chunks along with their embeddings

In [3]:
# -----------------------------------------------------------------------------------------------------------------
# Step 2: Generate Embeddings of all the documents using the `HuggingFaceEmbeddings` class that wraps Sentence Transformers (from Hugging Face)
# ------------------------------------------------------------------------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings # HuggingFaceEmbeddings class provides an interface to use pre-trained Hugging Face sentence-transformer models for generating text embeddings


# Initialize the Local Embedding Model using the `HuggingFaceEmbeddings` class that wraps Sentence Transformers (from Hugging Face)
# The returned object provides a simple interface to generate embeddings of text that can later be stored in a vector db or used for similarity search.
local_embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")      #"all-mpnet-base-v2"


# ------------------------------------------------------------------------------------------------------------------------------------------
# Step 3 (Option A): Create an In Memory Database using Chroma, add the documents to it, and display the doocuments along with their embeddings
# --------------------------------------------------------------------------------------------------------------------------------------------

from langchain_chroma import Chroma # Import Chroma from langchain_chroma (newer, recommended import)


# Chroma class is a vector store wrapper in LangChain for ChromaDB. It provides a high-level Python interface for storing document embeddings, performing similarity searches and integrating with LLM pipelines
vectorstore = Chroma(
    collection_name = "rag_collection",      # Name of the collection inside Chroma
    embedding_function = local_embedder,     # The embedding model used to generate vector representations
    collection_metadata={"hnsw:space": "cosine"}  # For Euclidean (L2) distance: "l2", For Inner product: "ip"
)

vectorstore.reset_collection() 
vectorstore.add_texts(texts = chunks)

#data = vectorstore.get()                      # returns a dictionary containing 'ids', 'embeddings', 'documents', and 'metadatas'
data = vectorstore.get(include=["documents", "metadatas", "embeddings"]) # returns a dictionary containing 'documents', 'metadatas', and  `embeddings`

print("\033[1m=== Displaying stored documents ===\033[0m")
for i, (doc, meta, emb) in enumerate(zip(data["documents"], data["metadatas"], data["embeddings"]),start=1):
    print(f"Document {i}")
    print(f"Metadata: {meta}")
    print(f"Content: {doc}")
    print(f"Embedding (first 5 values): {emb[:5]}...")
    print("-" * 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== Displaying stored documents ===
Document 1
Metadata: None
Content: bismillah ar-rahman rahim assalamu alaikum dear students I welcome you to this series of lectures for linux shell commands these are long awaited by my students operating system as well as system programming the main objective of the series of lectures is to acquaint the students of undergraduate operating system course to map the concepts that to study in the class using Linux tools it is also useful for the students who are learning Linux as system administrator students in this course in this series of lectures basically we will start with the concepts of UNIX shell then we will see how processes are managed by the UNIX shell how processes are created how the priorities are managed how they are scheduled how they are terminated we will see in detail how information about all the running processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico

## Step 4 (Query Processing): Create a Retriever and pass the User Query (string) to its `invoke()` method
- Retrieve top-k most relevant Chunks from the vector store 

In [4]:
query = "How binary software packages are installed?"

# Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",           # Type of search to perform: 'similarity', 'mmr', etc.
    search_kwargs={"k": 4},             # Dictionary of search parameters, default is none
    tags=None,             # List of tags for tracing/monitoring
    metadata=None,         # Dictionary of metadata
    verbose=False          # Enable verbose logging
)

results = retriever.invoke(query)     # The retriever.invokesimilarity_search() returns list containing Document objects only
unembedded_texts = []   # Initialize an empty list to store plain text from the retrieved Document objects
print(f"\n🔍 Query: '{query}'")
# Display results
for i, doc in enumerate(results, 1):
    print(f"\nContent {i}: {doc.page_content[:300]}...")
    unembedded_texts.append(doc.page_content)        # Add it to our list for potential later use
#print(unembedded_texts)


🔍 Query: 'How binary software packages are installed?'

Content 1: processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux particularly them about which Dennis eg has said if you do not know when you do not know UNIX we will see how process carries out inter process communic...

Content 2: bismillah ar-rahman rahim assalamu alaikum dear students I welcome you to this series of lectures for linux shell commands these are long awaited by my students operating system as well as system programming the main objective of the series of lectures is to acquaint the students of undergraduate op...

Content 3: Red Hat we will talk about different tools and commands that are used to manage the network on Linux and finally we will see a bit of details as how we can use these tools in bash shell scripting and if time permits I will of course talk about the Python as well last but not the least in between I w

## Step 5 (Context Augmentation): Build a prompt using the original query + top-K texts

In [5]:
# Step 1: Build the context section for the LLM prompt
# Combine the top retrieved text chunks (unembedded_texts) into a single context block that will guide the LLM’s response. Each chunk is labeled so the model can reference them (1/2/3).
context_parts = []
for i, t in enumerate(unembedded_texts, start=1):
    context_parts.append(f"--- Retrieved Document {i} ---\n{t}\n") # Add each retrieved document with a label for easy referencing

# Join all retrieved document texts into one large string, separated by line breaks for clarity.
context = "\n".join(context_parts).strip()


# Step 2: Construct the full prompt for the language model containing system role, context, user query and specific instructions as to how the model should respond
prompt = (
    "You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.\n\n"
    f"CONTEXT:\n{context}\n\n"        # Inject the retrieved document text here
    f"USER QUERY:\n{query}\n\n"    # The user's original question
    "INSTRUCTIONS: Answer concisely and cite which retrieved document (1/2/3) you used for any factual statement if relevant.\n\nAnswer:"
)


# Step 3: Display the constructed prompt for review
print(prompt)

You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.

CONTEXT:
--- Retrieved Document 1 ---
processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux particularly them about which Dennis eg has said if you do not know when you do not know UNIX we will see how process carries out inter process communication using signals pipes fie phones message queues and shared memory we will talk about the limitation of processes that make us move on to threads you can see special files like character special files block special files will see the terminals we will see how the stty command works we'll talk about user management in Linux we will talk about the permission sets of files and how they are implemented we will see how we can install binary software packages using apt yum or maybe rpm if we are using Red Hat we will talk about different tools and commands that are us

## Step 6 (Response Generation): Send the original query + top-K texts to an LLM of your Choice to generate answer using context

In [7]:
from dotenv import load_dotenv              # load_dotenv() method is used to securely load API keys and environment variables from a .env file
from langchain_openai import ChatOpenAI       # ChatOpenAI is LangChain wrapper for OpenAI-compatible chat models (also supports Groq and others)
import os

# Load environment variables from the .env file
load_dotenv('../keys/.env', override=True) 
groq_api_key = os.getenv('GROQ_API_KEY')

# Initialize the LLM (Groq-hosted model via OpenAI-compatible API)
model = ChatOpenAI(
    model="llama-3.3-70b-versatile",  
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",  # Point to Groq API
    temperature=0.7,                            # temperature: controls creativity (0.0 = deterministic, 1.0 = more creative)    
    max_tokens=512                              # max_tokens: limits the length of the generated response.
)

# Send the constructed prompt to the model using the .invoke() method that sends the text prompt to the model and returns a structured response object.
response = model.invoke(prompt) 

# Display the model's answer
print(response.content)    # The `content` attribute contains the actual text generated by the model.

Binary software packages are installed using apt, yum, or rpm (if using Red Hat) (Retrieved Document 1).


## Building a Chain

In [8]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [9]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [10]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [11]:
parallel_chain.invoke("How binary software packages are installed?")

{'context': "processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux particularly them about which Dennis eg has said if you do not know when you do not know UNIX we will see how process carries out inter process communication using signals pipes fie phones message queues and shared memory we will talk about the limitation of processes that make us move on to threads you can see special files like character special files block special files will see the terminals we will see how the stty command works we'll talk about user management in Linux we will talk about the permission sets of files and how they are implemented we will see how we can install binary software packages using apt yum or maybe rpm if we are using Red Hat we will talk about different tools and commands that are used to manage the network on Linux and finally we will see a bit of details as how we can use these tools in bash she

In [12]:
parser = StrOutputParser()

In [13]:
# Step 2: Construct the full prompt for the language model containing system role, context, user query and specific instructions as to how the model should respond
prompt = (
    "You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.\n\n"
    f"CONTEXT:\n{context}\n\n"        # Inject the retrieved document text here
    f"USER QUERY:\n{query}\n\n"    # The user's original question
    "INSTRUCTIONS: Answer concisely and cite which retrieved document (1/2/3) you used for any factual statement if relevant.\n\nAnswer:"
)


# Step 3: Display the constructed prompt for review
print(prompt)

You are a helpful assistant. Use ONLY the provided CONTEXT to answer the user's question.

CONTEXT:
--- Retrieved Document 1 ---
processes are managed by the Linux kernel inside the proc file system as well we will talk about different editors like vim Pico nano used by Linux particularly them about which Dennis eg has said if you do not know when you do not know UNIX we will see how process carries out inter process communication using signals pipes fie phones message queues and shared memory we will talk about the limitation of processes that make us move on to threads you can see special files like character special files block special files will see the terminals we will see how the stty command works we'll talk about user management in Linux we will talk about the permission sets of files and how they are implemented we will see how we can install binary software packages using apt yum or maybe rpm if we are using Red Hat we will talk about different tools and commands that are us

In [14]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [15]:
main_chain = parallel_chain | prompt | model | parser

In [16]:
answer = main_chain.invoke('Can you summarize the video')
print(answer)

The video is an introduction to a series of lectures on Linux shell commands. The lecturer welcomes students and explains that the series will cover various topics, including:

* UNIX shell concepts
* Process management (creation, priority, scheduling, termination)
* Information about running processes in the proc file system
* Editors like vim, Pico, and nano
* Inter-process communication (signals, pipes, etc.)
* Threads
* Special files
* Terminals and the stty command
* User management and file permissions
* Installing software packages
* Network management tools and commands
* Bash shell scripting
* Using GCC and gdb for system calls
* Possibly, Python programming (if time permits)

The lecturer aims to make the series informative and fun for learning Linux.


In [17]:
answer = main_chain.invoke('How binary softwares are installed?')
print(answer)

Binary software packages can be installed using apt, yum, or rpm (if using Red Hat).


In [18]:
answer = main_chain.invoke('What is the capital of Pakistan?')
print(answer)

I don't know.


In [19]:
answer = main_chain.invoke('What inter-process communication methods are discussed in the lecture?')
print(answer)

The lecture discusses the following inter-process communication methods: 

1. Signals
2. Pipes
3. Fifos
4. Message queues
5. Shared memory


In [20]:
answer = main_chain.invoke('Who developed the Linux kernel?')
print(answer)

I don't know.


In [21]:
answer = main_chain.invoke('What are the major topics discussed in this video? Just list them')
print(answer)

1. UNIX shell
2. Process management
3. Editors (vim, Pico, nano)
4. Inter-process communication
5. Threads
6. Special files
7. Terminals and stty command
8. User management
9. File permissions
10. Installing software packages
11. Network management
12. Bash shell scripting
13. System calls (using GCC and gdb)


In [22]:
answer = main_chain.invoke('Which programming tools are used to understand system calls like fork, wait, and exit?')
print(answer)

GCC and gdb are used to understand the usage of system calls like fork, wait, and exit.


In [23]:
answer = main_chain.invoke('What are character special files and block special files in Linux?')
print(answer)

The transcript doesn't provide a detailed explanation of character special files and block special files in Linux. It only mentions that they will be covered in the series of lectures. I don't know.


In [24]:
answer = main_chain.invoke('How does the Linux kernel manage information about running processes?')
print(answer)

The Linux kernel manages information about all the running processes inside the proc file system.


In [25]:
answer = main_chain.invoke('What is the difference between Linux and Windows process management?')
print(answer)

I don't know.


In [26]:
answer = main_chain.invoke('What is the name of the instructor?')
print(answer)

I don't know.


In [27]:
answer = main_chain.invoke('This course is designed for which students category?')
print(answer)

This course is designed for undergraduate students, particularly those studying operating system and system programming, as well as students learning Linux as system administrators.


# <span style='background :lightgreen' >5. RAG Types & Techniques</span>


## a. Core RAG Types

| **RAG Type** | **Key Features** | **When to Use** | **How It Works (Step-by-Step)** |
|--------------|------------------|-----------------|----------------------------------|
| **Naive/Standard RAG** | • Simple 3-step pipeline<br>• Direct vector similarity search<br>• No optimization or refinement<br>• Fixed chunking strategy | • Simple Q&A tasks<br>• Small, well-structured knowledge bases<br>• When speed matters more than accuracy<br>• Proof-of-concept projects | **Step 1:** Split documents into fixed-size chunks<br>**Step 2:** Convert chunks to embeddings and store in vector DB<br>**Step 3:** User asks question → convert to embedding<br>**Step 4:** Find top-K similar chunks via cosine similarity<br>**Step 5:** Pass retrieved chunks + query to LLM<br>**Step 6:** LLM generates answer |
| **Advanced RAG** | • Pre-retrieval optimization (query enhancement)<br>• Post-retrieval refinement (re-ranking)<br>• Smart chunking strategies<br>• Metadata enrichment | • Production applications<br>• Large, diverse knowledge bases<br>• When accuracy is critical<br>• Complex user queries | **Pre-Retrieval:**<br>**Step 1:** Optimize document indexing (hierarchical chunks, metadata tags)<br>**Step 2:** Enhance user query (expand, decompose, or rewrite)<br>**Step 3:** Perform retrieval with filters<br><br>**Post-Retrieval:**<br>**Step 4:** Re-rank results using cross-encoder<br>**Step 5:** Compress/filter irrelevant context<br>**Step 6:** Generate final answer with refined context |
| **Corrective RAG (CRAG)** | • Self-evaluation mechanism<br>• Automatic error correction<br>• Web search fallback<br>• Iterative refinement loop | • When knowledge base may be outdated<br>• Critical applications (medical, legal)<br>• Dynamic information needs<br>• When hallucinations must be minimized | **Step 1:** Retrieve documents (standard RAG)<br>**Step 2:** **Evaluator Agent** scores relevance of each document<br>**Step 3:** If relevance is HIGH → proceed to generation<br>**Step 4:** If relevance is LOW:<br>&nbsp;&nbsp;&nbsp;→ Rewrite query<br>&nbsp;&nbsp;&nbsp;→ Search external sources (web)<br>&nbsp;&nbsp;&nbsp;→ Merge internal + external results<br>**Step 5:** Filter out irrelevant parts<br>**Step 6:** Generate answer with corrected context |
| **[Agentic RAG](https://wandb.ai/byyoung3/Generative-AI/reports/Agentic-RAG-Enhancing-retrieval-augmented-generation-with-AI-agents--VmlldzoxMTcyNjQ5Ng)** | • Autonomous AI agents<br>• Dynamic decision-making<br>• Multi-step reasoning<br>• Tool use capabilities<br>• Self-reflection and planning | • Complex research tasks<br>• Multi-hop reasoning (requires connecting multiple facts)<br>• Dynamic environments<br>• When workflow needs to adapt based on intermediate results | **Step 1:** User query → **Planning Agent** breaks it into sub-tasks<br>**Step 2:** **Routing Agent** decides which knowledge sources to use<br>**Step 3:** Multiple retrieval operations based on plan<br>**Step 4:** **Reasoning Agent** analyzes retrieved info<br>**Step 5:** If incomplete → **Reflection Agent** identifies gaps<br>**Step 6:** Adaptive loop: re-plan and retrieve missing info<br>**Step 7:** **Synthesis Agent** combines all findings<br>**Step 8:** Generate comprehensive answer |
| **Speculative RAG** | • Dual-model architecture<br>• Parallel draft generation<br>• Verification mechanism<br>• Speed-accuracy balance | • Low-latency applications (chatbots)<br>• High-traffic systems<br>• When both speed AND accuracy matter<br>• Resource-constrained environments | **Step 1:** Retrieve relevant documents (standard)<br>**Step 2:** **Small Specialist Model** generates multiple draft answers in parallel (fast)<br>**Step 3:** Each draft uses slightly different retrieved contexts or interpretations<br>**Step 4:** **Large Generalist Model** evaluates all drafts<br>**Step 5:** Verifier checks accuracy, relevance, and completeness<br>**Step 6:** Select best draft or combine insights<br>**Step 7:** Return verified answer quickly |


## b.Advanced RAG Techniques

### (i) Pre-Retrieval Techniques (Happen BEFORE searching the knowledge base)

| **Technique** | **What It Does** | **Problem It Solves** | **Simple Example** |
|---------------|------------------|----------------------|-------------------|
| **Small-to-Big Chunking** | Index small chunks for precise retrieval, but retrieve surrounding larger context for generation | Fixed chunks lose important context; retrieved snippets too small for LLM | **Small chunk (indexed):** "Python uses garbage collection"<br>**Big chunk (retrieved):** Full paragraph explaining memory management, GC algorithms, and when GC runs |
| **Metadata Injection** | Add tags (date, author, document type, category) to each chunk | Hard to filter by source, date, or topic; retrieves irrelevant documents | User asks: "What was our Q3 revenue?"<br>Metadata filter: `date=Q3-2024` AND `type=financial_report`<br>→ Only retrieves Q3 financial docs, ignoring Q1/Q2 |
| **Multi-Query Expansion** | LLM generates 3-5 alternative phrasings of user's question | User's question may miss relevant documents due to vocabulary mismatch | **Original:** "How to speed up Python code?"<br>**Expanded:**<br>• "Python performance optimization techniques"<br>• "Making Python programs run faster"<br>• "Python code profiling and efficiency"<br>→ Retrieve from all queries, then combine results |
| **Query Decomposition** | Break complex questions into simpler sub-questions | Single retrieval can't answer multi-hop questions | **Original:** "Compare Python and Java memory management differences"<br>**Decomposed:**<br>1. "How does Python handle memory?"<br>2. "How does Java handle memory?"<br>3. Retrieve for each, then compare |

### (ii) Post-Retrieval Techniques (Happen AFTER initial retrieval)

| **Technique** | **What It Does** | **Problem It Solves** | **Simple Example** |
|---------------|------------------|----------------------|-------------------|
| **Re-ranking** | Use a smarter model (cross-encoder) to re-score retrieved documents | Initial retrieval (bi-encoder) is fast but less accurate; top results may not be most relevant | **Initial retrieval (bi-encoder):**<br>Doc1: 0.82, Doc2: 0.80, Doc3: 0.78<br><br>**After re-ranking (cross-encoder):**<br>Doc3: 0.95 ⭐, Doc1: 0.71, Doc2: 0.65<br>→ Doc3 is actually most relevant! |
| **Context Compression** | Remove redundant/irrelevant sentences, keep only essential information | LLM context window is limited; too much noise reduces answer quality | **Retrieved:** 3 documents (2000 tokens)<br>**After compression:**<br>• Remove repeated information<br>• Extract only sentences mentioning "neural networks"<br>• Result: 600 tokens of highly relevant content |

## c. Re-Ranking Technique

<div style="text-align:center;">
    <img src="../images/reranking01.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Re-ranking is not a retriever type rather a post-retrieval stage that sits between your retriever and the LLM, acting as a precision filter on a pool of candidates that the retriever gathered for recall.</h3>




<div style="text-align:center;">
    <img src="../images/reranking02.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>


#### The two-stage mental model
- **Stage 1 — Retrieval (fast, broad):** Any retriever from your table fires here. Its job is *recall* — get the right documents in the candidate set, even if imprecisely ranked. You ask for top-K (say 100 docs) because the retriever's scoring (cosine similarity) is an approximation of relevance.
- **Stage 2 — Re-ranking (slow, precise):** A cross-encoder model (like Cohere Rerank, BGE-Reranker, or `ms-marco-MiniLM`) reads every `(query, document)` pair *jointly* — it sees both at the same time, allowing deep attention-based interaction between query tokens and document tokens. It produces a true relevance score and reorders the pool, from which you pass only top-N (10) to the LLM.

>- In LangChain, you implement this by wrapping any retriever with `ContextualCompressionRetriever` and using a `CohereRerank` or `FlashrankRerank` compressor — the compressor does the re-ranking job.


## d. Quick Decision Guidelines

**Choose your RAG type based on your needs:**

```
START HERE
    ↓
Do you need BASIC Q&A with small data?
    YES → Use NAIVE RAG
    NO ↓

Do you need PRODUCTION-READY with better accuracy?
    YES → Use ADVANCED RAG (with pre/post techniques)
    NO ↓

Is your knowledge base OUTDATED or UNRELIABLE?
    YES → Use CORRECTIVE RAG
    NO ↓

Do you need MULTI-STEP REASONING or COMPLEX TASKS?
    YES → Use AGENTIC RAG
    NO ↓

Do you need FAST RESPONSES without sacrificing accuracy?
    YES → Use SPECULATIVE RAG
```

# <span style='background :lightgreen' >6. Vector RAG vs Vectorless RAG</span>

### [Vectorless RAG Pageindex (Sep 2025)](https://pageindex.ai/blog/pageindex-intro)
### [Vectorless RAG Microsoft (Mar 2026)](https://techcommunity.microsoft.com/blog/azuredevcommunityblog/vectorless-reasoning-based-rag-a-new-approach-to-retrieval-augmented-generation/4502238)

- The two approaches one can use are **Vectorized RAG** and **Vectorless RAG**
    - Vector RAG trades **setup complexity and compute cost** for **semantic intelligence**.
    - Vectorless RAG trades **retrieval depth** for **simplicity and speed**.
- **When Non-Vector RAG Works Better**
    - 👍 Strong cases:
        - Legal / contracts (exact wording matters)
        - Logs / error codes (exact match needed)
        - Structured data (IDs, numbers)
        - Code search
    - 👎 Weak cases:
        - Synonyms (“car” vs “vehicle”)
        - Paraphrased queries
        - Multilingual semantic matching
- The key difference between the two approaches is *how* relevant content is retrieved before the LLM generates its answer.
- To get the best of both worlds, many production systems use a **hybrid approach**  (combining TFIDF or BM25 keyword retrieval with vector search techniques like cosine similarity or HNSW)

> https://techcommunity.microsoft.com/blog/azuredevcommunityblog/vectorless-reasoning-based-rag-a-new-approach-to-retrieval-augmented-generation/4502238

| Feature | 🔵 **Vector RAG** | 🟢 **Vectorless RAG** |
|---|---|---|
| **Core Idea** | Represent text as dense numerical vectors; retrieve by geometric proximity in embedding space | Retrieve text using classical information retrieval signals: keyword frequency, position, and structure |
| **Pipeline Steps** | Chunking → Embedding → Storing in Vector DB → User query embedding → Retrieval/Reranking → Generate response | Build JSON tree index → User query →  LLM reasons over tree → Retrieve exact sections → Generate response|
| **Retrieval Method** | Semantic similarity (cosine / dot-product distance) | Lexical matching: BM25, TF-IDF, keyword, or full-context scan |
| **Embedding Model Required?** | ✅ Yes — e.g., `text-embedding-ada-002`, `all-MiniLM-L6-v2` | ❌ No embedding model needed |
| **Vector Database Required?** | ✅ Yes — e.g., FAISS, Chroma, Qdrant, Pinecone | ❌ No — plain text index or in-memory search suffices |
| **Query Understanding** | Semantic — captures *meaning, intent, and paraphrases* | Literal — matches *exact or closely related terms* |
| **Handles Synonyms / Paraphrasing?** | ✅ Yes — "heart attack" matches "myocardial infarction" | ⚠️ Partial — only if terms overlap lexically |
| **Handles Exact Keyword Lookups?** | ⚠️ Sometimes — can over-generalise | ✅ Yes — strong at precise term retrieval |
| **Indexing Overhead** | High — requires embedding every chunk before use | Low — lightweight or none |
| **Query-time Speed** | Fast (after indexing is complete) | Faster — no embedding step at query time |
| **Scalability to Large Docs** | ✅ Excellent — scales via ANN (Approximate Nearest Neighbour) search | ⚠️ Moderate — full-context scan slows on very large documents |
| **Setup Complexity** | Higher — chunking strategy, embedding model, vector DB, retrieval tuning | Lower — simpler architecture, fewer components |
| **Chunking Sensitivity** | ⚠️ High — poor chunk boundaries can break context and hurt retrieval | ✅ Low — operates on full sections or pages |
| **Hallucination Risk** | 🟡 Low — grounded in retrieved chunks, but chunk quality matters | 🟡 Low — grounded in source text, but may miss semantically relevant passages |
| **Best For** | Long documents, conceptual Q&A, cross-document synthesis, research assistants | Exact lookups, structured documents, short docs, compliance/legal Q&A |
| **Main Weakness** | Chunking can split important context; embedding cost; harder to debug | Fails on paraphrased or semantically varied queries; vocabulary mismatch |
| **Cost** | Higher — embedding APIs or GPU resources required | Lower — runs on CPU with minimal dependencies |
| **Free Online Tool to Try** | 🔗 [notebooklm.google.com](https://notebooklm.google.com) | 🔗 [chat.pageindex.ai](https://chat.pageindex.ai) |

> **Upload  same PDF [HO 1.4 (Recap of Internetworking Concepts with Linux](https://www.arifbutt.me/wp-content/uploads/2025/04/Handout-1.4-Recap-of-InterNetworking-Concepts-with-Linux.pdf) to both tools and ask different questions:**
>- "List the names of the attacks that can be launched on Internet layer"
>- "What port number does the web server listen on, and what is the exact subnet mask used in the VM networking example?" (Vector RAG might fail as Numbers like 80, 255.255.255.0 have no semantic embedding — cosine similarity won't reliably surface them)
>- "What is the exact Linux command used to configure the BIND DNS server?" (Vector RAG might fail as commands like` named.conf` or `rndc reload` are lexically precise; semantic embeddings blur them with loosely related content)
>- "List all TCP/IP stack layers mentioned in the document along with their corresponding protocols." (Vector RAG might fail as table rows are split into chunks during vectorization, losing row-column relationships)
>- "How does the firewall configuration relate to the client-server services described — which ports are explicitly allowed?" (Vector RAG might fail as this query requires combining facts from two distant sections; single-chunk retrieval misses the connection)
>- "What is the third step in setting up the virtual machine network interface?" (Vector RAG might fail as ordered steps get reranked by semantic relevance, not by their original sequence)
>- "What does the network topology diagram in the document illustrate about VM interconnections?" (Vector RAG might fail as images and figures are typically ignored during embedding; PageIndex can retrieve and render them directly)
